<a href="https://colab.research.google.com/github/Jennyaorozco/Laboratorios-ML/blob/main/Copia_de_lab4_parte1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 4 - Parte 1. Redes neuronales artificiales - perceptrón multicapa

In [1]:
!wget -nc --no-cache -O init.py -q https://raw.githubusercontent.com/jdariasl/Intro_ML_2025/master/init.py
import init; init.init(force_download=False); init.get_weblink()

replicating local resources
replicating Labs resources


In [4]:
from local.lib.rlxmoocapi import submit, session
import inspect

session.LoginSequence(
    endpoint=init.endpoint,
    course_id="introml",
    session_id="udea.vir.20262.g1",
    lab_id="L04.01",
    varname="student"
)

logging in as jennya.orozco@udea.edu.co... please wait
you are registered in the following sessions of this course

     ['udea.vir.20261', 'udea.vir.20262.g1']

and you have explicitly requested session 'udea.vir.20262.g1'.
your grades will be reported in that session

-------------
using course session introml::udea.vir.20262.g1
success!! you are logged in
-------------


In [5]:
#configuración del laboratorio
# Ejecuta esta celda!
from Labs.commons.utils.lab4 import *
_ = part_1()

Este ejercicio tiene como objetivo implementar una red neuronal artificial de tipo perceptrón multicapa (MLP) para resolver un problema de regresión. Usaremos la librería sklearn. Consulte todo lo relacionado con la definición de hiperparámetros, los métodos para el entrenamiento y la predicción de nuevas muestras en el siguiente enlace: http://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html#sklearn.neural_network.MLPRegressor

Para este ejercicio usaremos la base de datos sobre calidad del aire, que ha sido usada en laboratorios previos, pero en este caso trataremos de predecir dos variables en lugar de una, es decir, abordaremos **un problema de múltiples salidas**.

In [6]:
#cargamos la bd que está en un archivo .data y ahora la podemos manejar de forma matricial
db = np.loadtxt('Labs/commons/utils/data/AirQuality.data',delimiter='\t')  # Assuming tab-delimiter

#Esta es la base de datos AirQuality del UCI Machine Learning Repository. En el siguiente URL se encuentra toda
#la descripción de la base de datos y la contextualización del problema.
#https://archive.ics.uci.edu/ml/datasets/Air+Quality#

x = db[:,0:11]
y = db[:,11:13]

Para calcular los errores, vamos a explorar y usar el [modulo de metricas de sklearn](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics).

Podemos observar que el modulo tiene metricas para regresión y clasificación.

## Ejercicio 1 - Experimentar con MLP para regresión

Para porder implementar nuestra función, lo primero que debemos entender es la estructura de la red.

Como mencionamos, vamos a solucionar un problema de multiples salidas. Estas salidas contienen valores continuos. Por lo tanto debemos garantizar que la capa de salida de nuestra red tenga la capacidad de modelar este tipo de datos.


In [ ]:
#@title Pregunta Abierta
#@markdown ¿De acuerdo al problema planteado, que función de activación debe usar el MLP para un problema de regresión?
respuesta = "" #@param {type:"string"}

Una característica de los modelos de sklearn, es que ciertos tipos de atributos solo pueden ser accedidos cuando se entrena el modelo. Vamos a desarrollar una pequeña función para comprobar cuál es la función de activación de los modelos MLP para regresión de sklearn.

In [7]:
# ejercicio de código
def output_activation():
    """funcion que entrena un modelo
    con datos aleatorios para confirmar la función
    de activación de la ultima capa
    """
    mlp = MLPRegressor()
    # fit with some random data
    xrandom = np.random.rand(10,2)
    yrandom = np.zeros(10)
    # llamar el metodo adecuado para entrenar
    # el mlp con los x y 'y' random
    mlp.fit(xrandom, yrandom)
    # retornar el atributo de mlp adecuado
    return (mlp.out_activation_)

**Registra tu solución en línea**

In [ ]:
student.submit_task(namespace=globals(), task_id='T1');

In [ ]:
print("la función de activación para un problema de regresion es:", output_activation())

la función de activación para un problema de regresion es: identity


## Ejercicio 2 Experimentar con MLP para regresión

Una vez comprobado que sklearn usa la función de activación correcta, vamos a crear la función para realizar los experimentos.

Completa la función con el código necesario para usar una red neuronal tipo MLP para solucionar el problema de regresión propuesto.
1. Como función de activación en las capas ocultas use la función 'tanh'.
2. Ajuste el número máximo de épocas a 300.
3. Dejamos como variables el número de capas ocultas y el número de neuronas por capa
5. debemos seleccionar la función adecuada del [modulo de sklearn para calcular el Error Porcentual Absoluto Medio (MAPE en sigla en ingles)](https://scikit-learn.org/stable/modules/classes.html?highlight=metrics#module-sklearn.metrics). Tener en cuenta qué parametros usar.
6. Tenga en cuenta que para instanciar los modelos no se usan parámetros posicionales.

In [8]:
def experimentar_mlpr(num_hidden_layers, num_neurons, X,Y):
    """ función para realizar experimentos con el MLP
    num_hidden_layers: list de enteros con el número de capdas
        ocultas a usar
    num_neurons: list de enteros con el número de neuronas a usar
    X: matriz de numpy con los datos de entrada [muestras,variables]
    Y: matriz numpy con las variables a predecir

    Retorna: dataframe con 6 columnas:
        - número de capas, número de neuronas
        - promedio de error prueba variable 1 y desviación estandar
        - promedio de error prueba variable 2 y desviación estandar

    """
    #Validamos el modelo
    Folds = 3
    ss = ShuffleSplit(n_splits=Folds, test_size=0.2, random_state=1)
    resultados = pd.DataFrame()
    idx = 0
    for hidden_layers in num_hidden_layers:
        for neurons in num_neurons:
            for j, (train, test) in enumerate(ss.split(X)):
                # para almacenar errores intermedios
                ErrorY1 = np.zeros(Folds)
                ErrorY2 = np.zeros(Folds)
                Xtrain= X[train,:]
                Ytrain = Y[train,:]
                Xtest = X[test,:]
                Ytest = Y[test,:]
                #Normalizamos los datos
                scaler = StandardScaler().fit(X= Xtrain)
                Xtrain = scaler.transform(Xtrain)
                Xtest = scaler.transform(Xtest)
                #Haga el llamado a la función para crear y entrenar el modelo usando los datos de entrenamiento
                #Ajuste la semilla del modelo a 1 para garantizar la reproducibilidad de los resultados.
                hidden_layer_sizes = tuple([neurons] * hidden_layers)
                # entrena el MLP
                mlp = MLPRegressor(hidden_layer_sizes=hidden_layer_sizes, activation='tanh', max_iter=300, random_state=1)
                mlp.fit(Xtrain, Ytrain)
                # Use para el modelo para hacer predicciones sobre el conjunto Xtest
                Yest = mlp.predict(Xtest)
                # Mida el MAPE para cada una de las dos salidas

                ErrorY1[j] = mean_absolute_percentage_error(Ytest[:,0], Yest[:,0])
                ErrorY2[j] = mean_absolute_percentage_error(Ytest[:,1], Yest[:,1])

            print(f'error para salida 1 = {np.mean(ErrorY1):.4f} +- {np.std(ErrorY1):.4f}')
            print(f'error para salida 2 = {np.mean(ErrorY2):.4f} +- {np.std(ErrorY2):.4f}')

            resultados.loc[idx,'capas ocultas'] = hidden_layers
            resultados.loc[idx,'neuronas en capas ocultas'] = neurons
            resultados.loc[idx,'error de prueba y1(media)'] = np.mean(ErrorY1)
            resultados.loc[idx,'intervalo de confianza y1'] = np.std(ErrorY1)
            resultados.loc[idx,'error de prueba y2(media)'] = np.mean(ErrorY2)
            resultados.loc[idx,'intervalo de confianza y2'] = np.std(ErrorY2)
            idx+=1
    return (resultados)

**Registra tu solución en línea**

In [9]:
student.submit_task(namespace=globals(), task_id='T2');

vamos a realizar los experimentos

In [11]:
# tarda unos minutos!!
resultados_mlpr = experimetar_mlpr(num_hidden_layers = [1,2,3], num_neurons  = [8,12,16], X=x, Y=y)

NameError: name 'experimetar_mlpr' is not defined

In [10]:
# ver los resultados.
import seaborn as sns
sns.relplot(data = resultados_mlpr,  x='neuronas en capas ocultas', y = 'error de prueba y1(media)', style= 'capas ocultas', kind = 'line')
sns.relplot(data = resultados_mlpr,  x='neuronas en capas ocultas', y = 'error de prueba y2(media)', style= 'capas ocultas', kind = 'line')
plt.show()

NameError: name 'resultados_mlpr' is not defined

In [ ]:
#@title Pregunta Abierta
#@markdown Qué ocurre si cambiamos el valor de random state? ¿deberían los resultados ser iguales?
respuesta = "" #@param {type:"string"}

## Ejercicio 3 Experimentar con MLP para *clasificación*

A continuación se leen los datos de un problema de clasificación. El problema corresponde a la clasifiación de dígitos escritos a mano. Usaremos únicamente 4 de las 10 clases disponibles. Los datos fueron preprocesados para reducir el número de características.

In [12]:
digits = load_digits(n_class=4)
#--------- preprocesamiento--------------------
pca = PCA(0.99, whiten=True)
data = pca.fit_transform(digits.data)
#---------- Datos a usar ----------------------
Xd = data
Yd = digits.target

In [ ]:
#@title Pregunta Abierta
#@markdown  ¿Qué tipo de función de activación debe usar el modelo en la capa de salida para un problema de clasificación?
respuesta = "" #@param {type:"string"}

como lo hicmos antes, vamos a comprobar con la libreria la función de activación

In [15]:
# ejercicio de código
def output_activation_MPC():
    """funcion que entrena un modelo
    con data aleatoria para confirmar la funcion
    de activacion de la ultima capa
    """
    mlp = MLPClassifier()
    # fit with some random data
    xrandom = np.random.rand(10, 2)
    yrandom = np.zeros(10)
    # llamar el metodo adecuado para entrenar
    # el mlp con los x y 'y' random
    mlp.fit(xrandom, yrandom)
    # retornar el atributo de mlp adecuado
    return mlp.out_activation_

In [ ]:
print("la función de activación para un problema de clasificación es:", output_activation_MPC())

la función de activación para un problema de clasificación es: logistic


Ahora en nuestro siguiente ejercicio vamos a implementar una red neuronal artificial de tipo perceptrón multicapa (MLP) para resolver un problema de clasificación. Usaremos la librería sklearn. Consulte todo lo relacionado con la definición de hiperparámetros, los métodos para el entrenamiento y la predicción de nuevas muestras en el siguiente enlace: http://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier


Vamos completar la función con el código necesario para usar una red neuronal tipo MLP para solucionar el problema de clasificación propuesto.
1. Como función de activación en las capas ocultas use la función relu.
2. Ajuste el número máximo de épocas a 350.
3. Dejamos como variables el número de capas ocultas y el número de neuronas por capa
5. Seleccione la función adecuada del [modulo de sklearn para calcular la exactitud del clasificador](https://scikit-learn.org/stable/modules/classes.html?highlight=metrics#module-sklearn.metrics). Note que, según la definición de la función, debe retornar el error.

In [13]:
# ejercicio de código
def experimentar_mlpc(X, Y, num_hidden_layers, num_neurons):
    Folds = 4
    skf = StratifiedKFold(n_splits=Folds)
    resultados = pd.DataFrame()
    idx = 0
    for hidden_layers in num_hidden_layers:
        for neurons in num_neurons:
            for j, (train, test) in enumerate(skf.split(X, Y)):
                Error = np.zeros(Folds)          # DENTRO del loop de folds
                Xtrain = X[train,:]
                Ytrain = Y[train]
                Xtest = X[test,:]
                Ytest = Y[test]
                scaler = StandardScaler().fit(X=Xtrain)
                Xtrain = scaler.transform(Xtrain)
                Xtest = scaler.transform(Xtest)

                hidden_layer_sizes = tuple([neurons] * hidden_layers)
                mlp = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes,
                                    activation='relu',
                                    max_iter=350,
                                    random_state=1)
                mlp.fit(Xtrain, Ytrain)
                Yest = mlp.predict(Xtest)
                Error[j] = 1 - accuracy_score(Ytest, Yest)

            print('error para configuracion de params = ' + str(np.mean(Error)) + '+-' + str(np.std(Error)))

            resultados.loc[idx,'capas ocultas'] = hidden_layers
            resultados.loc[idx,'neuronas en capas ocultas'] = neurons
            resultados.loc[idx,'error de prueba(media)'] = np.mean(Error)
            resultados.loc[idx,'intervalo de confianza'] = np.std(Error)
            idx += 1
    return (resultados)

**Registra tu solución en línea**

In [14]:
student.submit_task(namespace=globals(), task_id='T3');

In [ ]:
# tarda unos minutos!!
resultados_mlpc = experimetar_mlpc(X = Xd, Y=Yd, num_hidden_layers=[1,2,3], num_neurons=[12,16,20])

In [ ]:
# ver los resultados
# notar como las capas ocultas y el # de neuronas influyen
import seaborn as sns
sns.relplot(data = resultados_mlpc,  x='neuronas en capas ocultas', y = 'error de prueba(media)', style= 'capas ocultas', kind = 'line')
plt.show()

In [ ]:
#@title Pregunta Abierta
#@markdown ¿Cuántas neuronas en la capa de salida tiene el modelo? ¿Porqué debe tener ese número?
respuesta = "" #@param {type:"string"}

## Ejercicio 4. Usar PyTorch para definir una arquitectura de red para clasificación.

Como se discutió en clase, los frameworks actuales para el diseño de redes neuronales artificales permiten mucha mayor flexibilidad a la hora de crear redes con diversas arquitecturas, diseñar funciones de coste o incluso nuevos tipos de capas. Uno de esos frameworks es [PyTorch](https://pytorch.org/).

Una red neuronal de tipo MLP es una red compuesta únicamente por capas Densas, llamadas en PyTorch capas lineales (`Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)).

- Lea la documentación del modulo [Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) de PyTorch
- Complete la siguiente función para crear un modelo de red neuronal para el problema de clasificación de digitos usando el modulo Sequential. Las funciones de activiación de las capas ocultas deben ser tipo 'relu'.
- Tenga en cuenta que Ud puede primero crear una lista de módulos (capas, activaciones, etc.) y luego crear el modelo `nn.Sequential(*modules)`. Las activaciones serán tratadas como capas adicionales.
- **No incluya** la función de activación de la capa de salida

In [16]:
#ejercicio de código
def MLP_classifier_pytorch(X,Y, neurons_per_layer):
  """
  Entrada:
  X: matriz de numpy con las muestras de entrada [muestras,variables]
  Y: vector numpy con las variables a predecir
  neurons_per_layer: list de enteros con el número de neuronas a usar por
  capa una de las capas ocultas

  Retorna:
  Modelo de PyTorch con la arquitectura deseada
  """

  import torch.nn as nn

  input_dim = X.shape[1]
  output_dim = len(np.unique(Y))

  modules = []
  previous_dim = input_dim
  for neurons in neurons_per_layer:
    modules.append(nn.Linear(previous_dim, neurons))
    modules.append(nn.ReLU())
    previous_dim = neurons

  modules.append(nn.Linear(previous_dim, output_dim))

  model = nn.Sequential(*modules)
  return model

**Registra tu solución en línea**

In [17]:
student.submit_task(namespace=globals(), task_id='T4');

In [18]:
#Experimentar con el modelo creado en PyTorch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import torch


#Data partition
train_indx, test_indx = train_test_split(np.arange(Xd.shape[0]), train_size=0.7, random_state=1)
Xtrain = Xd[train_indx,:]
Ytrain = Yd[train_indx]
Xtest = Xd[test_indx,:]
Ytest = Yd[test_indx]

#Normalization
scaler = StandardScaler().fit(X= Xtrain)
Xtrain = scaler.transform(Xtrain)
Xtest = scaler.transform(Xtest)

#Tensor transformation
Xtrain = torch.tensor(Xtrain, dtype=torch.float32)
Ytrain = torch.tensor(Ytrain, dtype=torch.long)
Xtest = torch.tensor(Xtest, dtype=torch.float32)

#Set the loss function
#En PyTorch la función de crossentropia aplica la activación Softmax, por eso no
# se debe incluir en la construcción del modelo.
loss_fn = torch.nn.CrossEntropyLoss()

model = MLP_classifier_pytorch(X = Xtrain, Y=Ytrain, neurons_per_layer=[20,10])

#Definimos el optimizador
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

max_epochs = 100
model.train()
for i in range(max_epochs):
  out_put = model(Xtrain)
  loss = loss_fn(out_put, Ytrain)
  print(f'epoch = {i}, loss = {loss}')
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

model.eval()
Yest = model(Xtest)
Yest = torch.argmax(Yest, axis=1)
print(f'Test accuracy = {accuracy_score(Ytest, Yest)}')

epoch = 0, loss = 1.4077450037002563
epoch = 1, loss = 1.4031099081039429
epoch = 2, loss = 1.3947103023529053
epoch = 3, loss = 1.3834894895553589
epoch = 4, loss = 1.370436191558838
epoch = 5, loss = 1.35639226436615
epoch = 6, loss = 1.3415755033493042
epoch = 7, loss = 1.3259574174880981
epoch = 8, loss = 1.309110403060913
epoch = 9, loss = 1.2901328802108765
epoch = 10, loss = 1.2679393291473389
epoch = 11, loss = 1.2412893772125244
epoch = 12, loss = 1.2088227272033691
epoch = 13, loss = 1.1693718433380127
epoch = 14, loss = 1.1224325895309448
epoch = 15, loss = 1.0680919885635376
epoch = 16, loss = 1.0072263479232788
epoch = 17, loss = 0.9413832426071167
epoch = 18, loss = 0.8723562359809875
epoch = 19, loss = 0.8018797636032104
epoch = 20, loss = 0.7309073805809021
epoch = 21, loss = 0.6605072617530823
epoch = 22, loss = 0.5932260751724243
epoch = 23, loss = 0.532242476940155
epoch = 24, loss = 0.4790451228618622
epoch = 25, loss = 0.430965781211853
epoch = 26, loss = 0.3847503